# Custom Dataset YOLOv11 Fine-tuning & ONNX Export
본 노트북은 n개의 클래스로 구성된 640x640 크기의 이미지를 학습하고, 완료 후 ONNX 모델로 내보내는 파이프라인을 포함하고 있습니다.

## 1. 구글 드라이브 마운트

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab = True
except:
    print('local drive.')
    colab = False

Mounted at /content/drive
g-drive mounted.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. 경로 설정 및 데이터셋 준비
환경에 맞게 `data.yaml`의 경로를 수정해 주세요. 본 예시는 기존 노트북 구조를 유지하여 작성되었습니다.

In [5]:
import os

if colab:
    # 코랩 환경에서의 경로 예시
    DATA_YAML = '/content/drive/MyDrive/Classroom/Project_TEAM6/data.yaml'  # data.yaml 경로
    project_root = '/content/drive/MyDrive/Classroom/Project_TEAM6'   # project root 경로
else:
    # 로컬 환경에서의 경로 예시
    DATA_YAML = './dataset/data.yaml'
    project_root = './custom_runs'

# unzip
!unzip -q /content/drive/MyDrive/Classroom/Project_TEAM6/Dataset.zip -d /content  # dataset.zip unzip
!unzip -o "/content/drive/MyDrive/Classroom/Project_TEAM6/last.pt" -d "/content"  # last.pt unzip

print(f"사용 예정 data.yaml 경로: {DATA_YAML}")

Archive:  /content/drive/MyDrive/Classroom/Project_TEAM6/last.pt
 extracting: /content/archive/data.pkl  
 extracting: /content/archive/.format_version  
 extracting: /content/archive/.storage_alignment  
 extracting: /content/archive/byteorder  
 extracting: /content/archive/data/0  
 extracting: /content/archive/data/1  
 extracting: /content/archive/data/2  
 extracting: /content/archive/data/3  
 extracting: /content/archive/data/4  
 extracting: /content/archive/data/5  
 extracting: /content/archive/data/6  
 extracting: /content/archive/data/7  
 extracting: /content/archive/data/8  
 extracting: /content/archive/data/9  
 extracting: /content/archive/data/10  
 extracting: /content/archive/data/11  
 extracting: /content/archive/data/12  
 extracting: /content/archive/data/13  
 extracting: /content/archive/data/14  
 extracting: /content/archive/data/15  
 extracting: /content/archive/data/16  
 extracting: /content/archive/data/17  
 extracting: /content/archive/data/18  
 ex

### 2-1. xml to txt convert

In [6]:
# 클래스 이름 순서 (data.yaml과 반드시 일치해야 함)
# CLASSES = ['크라운쵸코하임284G', '농심벌집핏자90G', '농심포스틱84G', '빙그레꽃게랑오리지널맛70G', '꽃게랑불짬뽕맛70G', '꽃게랑와사비70G' ]

import csv

csv_file_path = '/content/drive/MyDrive/Classroom/Project_TEAM6/class.csv'  # class.csv 경로
CLASSES = []
with open(csv_file_path, mode='r', encoding='cp949') as f:
  reader = csv.reader(f)

  next(reader, None)

  for row in reader:
    if len(row) > 1:
      CLASSES.append(row[1].strip())

In [7]:
CLASSES

['롯데칠성사이다245ML',
 '오)포카칩어니언맛110g',
 '포카칩오리지널110G',
 '오리온)왕꿈틀이67G',
 '오리온마이구미1P66G',
 '크라운초코하임47G',
 '크라운화이트하임284G',
 '한국마즈)이클립스스트로베리향',
 '한국마즈)이클립스스피어민트향',
 '한국마즈)이클립스페퍼민트향',
 '한국마즈)이클립스피치향',
 '오리온썬핫스파이시맛80G',
 '오리온스윙칩볶음고추장60G',
 '롯데펩시콜라600ML',
 '코카콜라제로500ML']

In [8]:
import xml.etree.ElementTree as ET
import os

dataset_root = "/content/Dataset"

def convert(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[2]) / 2.0
    y = (box[1] + box[3]) / 2.0
    w = box[2] - box[0]
    h = box[3] - box[1]
    return (x * dw, y * dh, w * dw, h * dh)

def convert_folder(src_folder, dst_labels_folder):
    # 하위 폴더 경로 구조 그대로 생성 (예: labels/30060)
    if not os.path.exists(dst_labels_folder):
        os.makedirs(dst_labels_folder)

    xml_files = [f for f in os.listdir(src_folder) if f.endswith('.xml')]
    if not xml_files:
        return

    for filename in xml_files:
        in_file_path = os.path.join(src_folder, filename)
        out_file_path = os.path.join(dst_labels_folder, filename.replace('.xml', '.txt'))

        tree = ET.parse(in_file_path)
        root = tree.getroot()

        size = root.find('size')
        if size is None: continue
        w = int(size.find('width').text)
        h = int(size.find('height').text)

        with open(out_file_path, 'w', encoding='utf-8') as out_file:
            for obj in root.iter('object'):
                cls = obj.find('name').text
                if cls not in CLASSES:
                    continue
                cls_id = CLASSES.index(cls)

                xmlbox = obj.find('bndbox')
                b = (float(xmlbox.find('xmin').text),
                     float(xmlbox.find('ymin').text),
                     float(xmlbox.find('xmax').text),
                     float(xmlbox.find('ymax').text))

                bb = convert((w, h), b)
                out_file.write(f"{cls_id} {' '.join([f'{a:.6f}' for a in bb])}\n")

# === 하위 디렉토리 구조를 그대로 유지하며 변환 ===
sub_modes = ["Training", "Validation"]

for mode in sub_modes:
    base_label_dir = os.path.join(dataset_root, mode, "label")

    if os.path.exists(base_label_dir):
        sub_dirs = [d for d in os.listdir(base_label_dir) if os.path.isdir(os.path.join(base_label_dir, d))]

        for sub_dir in sub_dirs:
            src_folder = os.path.join(base_label_dir, sub_dir)
            # 💡 dst 경로에 sub_dir을 유지하여 'labels/30060' 형태로 저장되도록 함
            dst_labels_folder = os.path.join(dataset_root, mode, "labels", sub_dir)

            convert_folder(src_folder, dst_labels_folder)
        print(f"변환 완료: {mode} 완료")

변환 완료: Training 완료
변환 완료: Validation 완료


## 3. Ultralytics 패키지 설치

In [9]:
!pip -q install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 66.7 MB/s eta 0:00:00


In [10]:
import torch

# last.pt 파일 경로 (본인 드라이브 경로에 맞게 수정)
ckpt_path = '/content/drive/MyDrive/Classroom/Project_TEAM6/last.pt'  # last.pt 경로

try:
    # 모델 가중치 파일 열기 (weights_only=False 필수)
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

    # 딕셔너리에서 'epoch' 키값 가져오기
    epoch = ckpt.get('epoch')

    if epoch is not None:
        # epoch은 0부터 시작하므로 +1을 해줘야 실제 횟수가 됩니다.
        print(f"✅ 현재까지 완료된 에포크: {epoch + 1} 번")
    else:
        print("⚠️ 에포크 정보를 찾을 수 없습니다.")

except Exception as e:
    print(f"❌ 에러 발생: {e}")
    print("파일 경로가 정확한지 다시 한번 확인해 주세요!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ 현재까지 완료된 에포크: 70 번


## 4. 모델 로드 및 학습 진행 (640*640 설정)
n개의 클래스를 인식하는 `data.yaml` 정보와 함께 이미지 크기를 **640**으로 설정하여 학습을 진행합니다.

In [ ]:
from ultralytics import YOLO

LAST_PT_PATH = '/content/drive/MyDrive/Classroom/Project_TEAM6/last.pt' # last.pt의 경로
# (중요!) last.pt가 있는 폴더 안에 args.yaml 파일도 반드시 있어야함!(args.yaml 파일은 학습 시 생성됨)
model = YOLO(LAST_PT_PATH)

print("🚀 중단된 모델을 불러와 이어서 학습을 시작합니다!")

# 2. 이어서 학습
results = model.train(
    resume=True,
    data=DATA_YAML,
    optimizer='SGD'
)

🚀 중단된 모델을 불러와 이어서 학습을 시작합니다!
Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.2, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Classroom/Project_TEAM6/data.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.2, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.02, hsv_s=0.6, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=/content/drive/MyDrive/Classroom/Project_TEAM6/last.pt, momentum=0.937, mosaic=0.8, multi_scale=0.0, name=P

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/plotting.py:656: UserWarning: Glyph 47215 (\N{HANGUL SYLLABLE ROS}) missing from font(s) DejaVu Sans.
  plt.savefig(fname, dpi=200)
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/plotting.py:656: UserWarning: Glyph 45936 (\N{HANGUL SYLLABLE DE}) missing from font(s) DejaVu Sans.
  plt.savefig(fname, dpi=200)
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/plotting.py:656: UserWarning: Glyph 52832 (\N{HANGUL SYLLABLE CIL}) missing from font(s) DejaVu Sans.
  plt.savefig(fname, dpi=200)
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/plotting.py:656: UserWarning: Glyph 49457 (\N{HANGUL SYLLABLE SEONG}) missing from font(s) DejaVu Sans.
  plt.savefig(fname, dpi=200)
/usr/local/lib/python3.12/dist-packages/ultralytics/utils/plotting.py:656: UserWarning: Glyph 49324 (\N{HANGUL SYLLABLE SA}) missing from font(s) DejaVu Sans.
  plt.savefig(fname, dpi=200)
/usr/local/lib/python3.12/dist-packages/ultralytic

Resuming training /content/drive/MyDrive/Classroom/Project_TEAM6/last.pt from epoch 71 to 200 total epochs
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/drive/MyDrive/ai/project/03_result/0628/Project_TEAM6_trained
Starting training for 200 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     71/200      8.12G      0.379     0.2834     0.9412         18        640: 100% ━━━━━━━━━━━━ 400/400 1.5it/s 4:36
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.2s/it 10.0s
                   all        225        388      0.985      0.993      0.989      0.978

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     72/200       8.2G     0.3813     0.2825       0.94         27        640: 100% ━━━━━━━━━━━━ 400/400 1.5it/s 4:18
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━

## 5. 최적 성능 가중치 파일 로드 및 ONNX 변환(Export)

In [ ]:
# 학습 완료 후 검증 성능이 가장 좋았던 best.pt 경로 확보
best_model_path = os.path.join(model.trainer.save_dir, 'weights', 'best.pt')
best_model = YOLO(best_model_path)

print(f"최적 가중치 모델 로드 완료: {best_model_path}")

# ONNX 파일 형식으로 익스포트
# imgsz=640으로 지정하여 640x640 정적 입력 해상도를 고정할 수 있습니다.
onnx_path = best_model.export(format='onnx', imgsz=640)

print(f"ONNX 파일 내보내기 성공: {onnx_path}")

최적 가중치 모델 로드 완료: /content/drive/MyDrive/Classroom/Project_TEAM6/Project_TEAM6_trained_v2_resume/weights/best.pt
Ultralytics 8.4.78 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11m summary (fused): 126 layers, 20,041,597 parameters, 0 gradients, 67.7 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/Classroom/Project_TEAM6/Project_TEAM6_trained_v2_resume/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 19, 8400) (38.7 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 668ms
Prepared 4 packages in 2.12s
Installed 4 packages in 357ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate su